# v2 코호트: Swee 2026 기준 4년 정답 → 최종 사용 환자 445명

**목적**: Swee 외 2026의 코호트 규칙(`2_Labeling/Labeling_MUSIC_Updated.ipynb`)을 그대로 따라 **730명과 4년 SCD/PFD 정답**을 만든 뒤, 팀 분할 파일(`final_split.csv`, 508명)에서 **등록 전 기기 이식 환자를 빼고 최종 사용 환자 445명**을 정한다. train/val/test 분할은 팀 분할 파일을 그대로 따른다. v2 증강(`final_holter_ecg_2_augmentation.ipynb`)과 전처리(`final_holter_ecg_3_preprocessing.ipynb`)는 445명 표(`cohort_final.csv`)를 쓴다.

| 출처 | 링크 |
|---|---|
| 논문 | Swee S, et al. medRxiv 2026. [본문](https://www.medrxiv.org/content/10.64898/2026.04.21.26351421v2.full) |
| 코드 (GPL-3.0, 규칙만 보고 다시 작성) | [github.com/stswee/multimodal_aecg_clinicalfeatures](https://github.com/stswee/multimodal_aecg_clinicalfeatures) |
| 팀 분할 파일 | `results/final_holter_ecg/final_split.csv` (`pid`, `split`, `SCD`). 508명 |
| 대조용 | Swee GitHub 릴리스 `MUSIC_Analysis_Release_v2`의 ECG 예측 파일에서 뽑은 `results/final_holter_ecg/release_ecg_predictions.csv` (환자 목록, Swee ECG 단독 예측 확률) |

## 기준 (모두 Swee 코드 그대로)

| 순서 | 규칙 | 사용 열 |
|---|---|---|
| 1 | Holter 있음 | `Holter available` = 1 |
| 2 | 결과가 생존, SCD, 펌프부전 사망(PFD) 중 하나 (비심장사 제외) | `Cause of death` ∈ {0, 3, 6, 7} |
| 3 | 등록 전 이식형 심장기기 없음 | `Prior implantable device` = 0 |
| 4 | 심장이식으로 추적 종료되지 않음 | `Exit of the study` ≠ 2 |
| 5 | 4년 결과를 알 수 있음: 4년 이상 추적했거나, 4년 안에 SCD/PFD | `Follow-up period from enrollment (days)` |

- 4년 = **1,460일** (4 × 365)
- `SCD_4year_label`: 4년 안 SCD = 1, 심장사 없음 = 0, **4년 안 PFD = 빈 값** (SCD 분석에서 빠짐)
- `PFD_4year_label`: 반대로 4년 안 SCD = 빈 값
- `outcome_code`: 0 = 4년 안 심장사 없음, 3 = SCD, 6 = PFD. 분할할 때 층화 기준으로 쓸 수 있음 (Swee는 이 값으로 층화)
- 4년이 지난 뒤 SCD·PFD로 사망한 환자는 4년 정답이 0 (4년 동안은 사건 없음). Swee 코드도 같음 (`Follow-up ≤ 1,460일` 조건)

## 최종 사용 환자 (⑥)

| 순서 | 규칙 | 사용 열 |
|---|---|---|
| 6 | 팀 분할 파일에 있는 환자만 (508명) | `final_split.csv`의 `pid` |
| 7 | 등록 전 기기 이식 없음 → 445명 | `Prior implantable device` = 0 |

- 정답은 분할 파일의 `SCD` 열이 아니라 위의 `SCD_4year_label`을 쓴다. 분할 파일의 `SCD`는 추적 기간 전체 기준이라 4년 뒤 SCD 3명(P0043, P0346, P0539)이 1로 되어 있음 → 4년 정답에서는 0
- 미리 대조한 값 (⑥ 실행 결과로 확인): 기기 이식 63명 제외 (CRT-I 19, Pacemaker-III 18, Pacemaker-I 13, Pacemaker-II 7, CRT-II 6) → 445명, SCD 50명 · 4년 안 심장사 없음 395명. split별 train 280 (SCD 29) · val 74 (SCD 9) · test 91 (SCD 12)
- 분할 파일에는 PFD 환자가 없음 (4년 뒤 PFD 5명 포함). 730명 중 4년 SCD 21명과 생존 177명도 분할 파일에 없는데, 이 198명이 빠진 기준은 확인 중

## 결정 기록 (사용자가 정한 것만)

| 날짜 | 결정 |
|---|---|
| 2026-10-26 | v2는 Swee 2026 코호트(730명, 4년 정답)를 그대로 사용 |
| 2026-10-26 | 전처리 결과는 분할 없이 제공. train/val/test 분할은 사용자가 직접 함 |
| 2026-10-26 | 기존 `SCD_Holter_DL_10s`와 `split_team_v1`은 v2에서 쓰지 않음 |
| 2026-10-27 | 팀 분할 파일(508명)에서 등록 전 기기 이식 환자 63명을 빼고 **445명(4년 SCD 50명)으로 확정**. 정답은 `SCD_4year_label` |
| 2026-10-27 | 증강과 전처리도 445명(`cohort_final.csv`)만 대상으로 함 |

In [ ]:
# ① 임상 CSV 읽기: Swee와 같은 방식 (구분자 ';', 소수점 ',', 숫자 안의 탭을 '.'로)
from pathlib import Path

import pandas as pd

SUBJECT_CSV = Path("/mnt/c/data/music/subject-info.csv")
RELEASE_PATH = Path("../results/final_holter_ecg/release_ecg_predictions.csv")   # Swee 릴리스 대조용
COHORT_PATH = Path("../results/final_holter_ecg/cohort_4year.csv")                    # 이 노트북의 결과 (분할 없음)
FLOW_PATH = Path("../results/final_holter_ecg/flow_cohort.csv")
SPLIT_PATH = Path("../results/final_holter_ecg/final_split.csv")       # 팀 분할 파일 (pid, split, SCD)
FINAL_PATH = Path("../results/final_holter_ecg/cohort_final.csv")      # 최종 사용 환자 445명: split + 4년 정답

HORIZON_DAYS = 4 * 365       # 4년 = 1,460일

# 사망 원인 코드 (subject-info_codes.csv)
SURVIVOR, NON_CARDIAC, SCD = 0, 1, 3
PFD_CODES = [6, 7]
TRANSPLANT_EXIT = 2          # Exit of the study = 2 → 심장이식

subject = pd.read_csv(SUBJECT_CSV, sep=";", decimal=",", engine="python", na_values=["", "NA"])
subject = subject.replace(r"\t", ".", regex=True)
assert subject.shape == (992, 103), subject.shape          # Swee 코드의 확인 조건과 같음
print("임상 CSV:", subject.shape)

임상 CSV: (992, 103)


In [ ]:
# ② 코호트 5단계: 단계마다 몇 명이 남는지 표로 기록
follow_up = subject["Follow-up period from enrollment (days)"]
cause = subject["Cause of death"]
event_within_4y = cause.isin([SCD] + PFD_CODES) & follow_up.le(HORIZON_DAYS)

criteria = {
    "Holter 있음": subject["Holter available"].eq(1),
    "생존·SCD·PFD만 (비심장사 제외)": cause.isin([SURVIVOR, SCD] + PFD_CODES),
    "등록 전 기기 이식 없음": subject["Prior implantable device"].eq(0),
    "심장이식 아님": subject["Exit of the study"].ne(TRANSPLANT_EXIT) | subject["Exit of the study"].isna(),
    "4년 결과 확인 가능": follow_up.ge(HORIZON_DAYS) | event_within_4y,
}

keep = pd.Series(True, index=subject.index)
flow = [{"단계": "시작 (MUSIC 전체)", "남은 환자": len(subject), "제외": 0}]
for step, rule in criteria.items():
    before = int(keep.sum())
    keep &= rule.fillna(False)
    flow.append({"단계": step, "남은 환자": int(keep.sum()), "제외": before - int(keep.sum())})

flow = pd.DataFrame(flow)
display(flow)
cohort = subject.loc[keep].copy()

,단계,남은 환자,제외
0,시작 (MUSIC 전체),992,0
1,Holter 있음,936,56
2,생존·SCD·PFD만 (비심장사 제외),879,57
3,등록 전 기기 이식 없음,758,121
4,심장이식 아님,746,12
5,4년 결과 확인 가능,730,16


In [ ]:
# ③ 4년 정답: SCD와 PFD를 서로의 분석에서 빼는 방식 (경쟁 사건은 '0'이 아니라 빈 값)
scd_4y = (cohort["Cause of death"].eq(SCD) & cohort["Follow-up period from enrollment (days)"].le(HORIZON_DAYS))
pfd_4y = (cohort["Cause of death"].isin(PFD_CODES) & cohort["Follow-up period from enrollment (days)"].le(HORIZON_DAYS))

labels = pd.DataFrame({
    "pid": cohort["Patient ID"].str.strip().values,                  # 'P0001' 형식 (Swee는 'P'를 뗀 '0001')
    "SCD_4year": scd_4y.astype(int).values,
    "PFD_4year": pfd_4y.astype(int).values,
})
labels["SCD_4year_label"] = labels["SCD_4year"].where(labels["PFD_4year"].eq(0)).astype("Int64")
labels["PFD_4year_label"] = labels["PFD_4year"].where(labels["SCD_4year"].eq(0)).astype("Int64")
labels["outcome_code"] = 0                                          # 0 = 4년 안 심장사 없음
labels.loc[labels["SCD_4year"].eq(1), "outcome_code"] = 3
labels.loc[labels["PFD_4year"].eq(1), "outcome_code"] = 6

assert labels["pid"].is_unique and not (labels["SCD_4year"].eq(1) & labels["PFD_4year"].eq(1)).any()
print("4년 결과:", labels["outcome_code"].map({0: "심장사 없음", 3: "SCD", 6: "PFD"}).value_counts().to_dict())
print("SCD 분석 대상:", labels["SCD_4year_label"].notna().sum(), "명 (SCD", int(labels["SCD_4year_label"].sum()), "명)")

4년 결과: {'심장사 없음': 577, 'PFD': 82, 'SCD': 71}
SCD 분석 대상: 648 명 (SCD 71 명)


In [ ]:
# ④ Swee 릴리스와 환자 목록 대조 (분할은 하지 않음)
release = pd.read_csv(RELEASE_PATH)
check = labels.merge(release[["pid"]], on="pid", how="outer", indicator=True)
print("환자 목록이 Swee와 같은가:", (check["_merge"] == "both").all(), check["_merge"].value_counts().to_dict())

환자 목록이 Swee와 같은가: True {'both': 730, 'left_only': 0, 'right_only': 0}


In [ ]:
# ⑤ 저장: 환자 1줄 = 정답. 단계별 인원도 같이 저장
labels.to_csv(COHORT_PATH, index=False)
flow.to_csv(FLOW_PATH, index=False)
print("저장:", COHORT_PATH, len(labels), "명")
print("저장:", FLOW_PATH)

저장: ../results/preprocess_swee/cohort_4year.csv 730 명
저장: ../results/preprocess_swee/flow_cohort.csv


In [ ]:
# ⑥ 최종 사용 환자: 팀 분할 파일에서 등록 전 기기 이식 환자를 빼고, 정답은 ③의 4년 정답으로 바꿈
split = pd.read_csv(SPLIT_PATH)
split["pid"] = split["pid"].str.strip()
device = pd.DataFrame({"pid": subject["Patient ID"].str.strip(), "device": subject["Prior implantable device"]})
split = split.merge(device, on="pid", how="left", validate="one_to_one")
assert split["device"].notna().all(), "임상 CSV에 없는 환자가 분할 파일에 있음"

no_device = split["device"].eq(0)
print("분할 파일:", len(split), "명 · 기기 이식 제외", int((~no_device).sum()), "명",
      split.loc[~no_device, "device"].value_counts().to_dict())   # 코드: 1·2·3 = Pacemaker, 4·8 = CRT

final = split.loc[no_device].merge(labels[["pid", "SCD_4year_label", "outcome_code"]], on="pid", how="left", validate="one_to_one")
assert final["SCD_4year_label"].notna().all(), "730명 코호트 밖 환자나 PFD 환자가 남아 있음"
changed = final[final["SCD"] != final["SCD_4year_label"]]
print("정답이 바뀐 환자 (분할 파일 SCD → 4년 정답):", changed[["pid", "SCD", "SCD_4year_label"]].to_dict("records"))

final = final[["pid", "split", "SCD_4year_label", "outcome_code"]]
display(pd.crosstab(final["split"], final["SCD_4year_label"], margins=True))
final.to_csv(FINAL_PATH, index=False)
print("저장:", FINAL_PATH, len(final), "명 · SCD", int(final["SCD_4year_label"].sum()), "명")


📝 **결과 메모**
- 단계별 인원 (Swee 논문: Holter 없음 56, 결과 범주 밖 57, 기기 121, 이식 12, 4년 불확실 16 → 730):
- 4년 결과 (Swee: 심장사 없음 577, SCD 71, PFD 82):
- Swee 릴리스와 환자 목록 일치 여부:
- ⑥ 분할 파일 인원 → 기기 이식 제외 → 최종 인원, split별 SCD (미리 대조한 값: 508 → 63 제외 → 445, SCD 50):
- ⑥ 정답이 바뀐 환자 (예상: P0043, P0346, P0539):